# Cross-Lingual Spatial Mathematics → Braille + Audio Assistant

An intelligent multimodal pipeline that lets a blind student ask a math question by 
**typing, speaking, or photographing a textbook page**, and get back a short 
step-by-step solution as an on-screen Braille preview plus slow, localized audio — 
optionally translated into another Indian language along the way.

**Hybrid architecture:** Student Input → Transcription (local faster-whisper) / OCR 
(OpenCV + pytesseract) → Data Integrity Checks → **local deterministic Spatial 
Mathematics Decoder (engine.py) tried first** → openai/gpt-oss-20b (via Groq) used 
*only* for translating an already-solved answer or as a clearly-flagged fallback for 
problems the local engine can't parse → Split-Track Output (Braille + Audio).

This hybrid design means every answer the local engine can solve (arithmetic, linear/
quadratic equations, matrices) is **guaranteed correct and works fully offline** — the 
cloud LLM never touches the actual math, only translation or explicit best-effort 
fallback, which is always labeled as such in the output.


## 0. Setup


In [ ]:
!pip install -q groq pytesseract Pillow opencv-python-headless numpy faster-whisper gTTS
!apt-get install -y tesseract-ocr -qq  # Colab only; on macOS use `brew install tesseract`


In [ ]:
import os
# Paste your free Groq API key (https://console.groq.com/keys) or set it as a Colab secret.
# Only needed for translation into non-native languages or fallback solving --
# locally-solvable questions in English/Hindi/Tamil work with NO key and NO internet.
os.environ['GROQ_API_KEY'] = os.environ.get('GROQ_API_KEY', '')


In [ ]:
import sys
sys.path.insert(0, '.')
from pipeline import (
    ocr_accuracy, ocr_confidence, ocr_image, transcribe_voice,
    text_to_braille, braille_accuracy, synthesize_audio, run_pipeline,
)
import spatial_decoder as sd


## 1. Student Input Options (Section 4.1)

Run the cell below and choose:

```
1 for Type Question
2 for voice upload
3 for image upload
```

*(Take a screenshot of this cell's output for your writeup's Section 4.1 figure.)*


In [ ]:
from google.colab import files  # Colab file-picker; comment out if running locally

print('1 for Type Question, 2 for voice upload, 3 for image upload')
choice = input('Choose: ').strip()

question_text = None

if choice == '1':
    question_text = input('Type your math question: ')

elif choice == '2':
    uploaded = files.upload()  # upload a .wav voice note
    wav_path = next(iter(uploaded))
    source_lang = input('Voice note language code (en/hi/ta/te/kn/mr/bn/gu): ') or 'hi'
    question_text = transcribe_voice(wav_path, language=source_lang)  # offline, faster-whisper
    print('Transcribed:', question_text)

elif choice == '3':
    uploaded = files.upload()  # upload a textbook photo (.png/.jpg)
    image_path = next(iter(uploaded))
    extracted = ocr_image(image_path)  # OpenCV-preprocessed + pytesseract
    print('OCR extracted:', extracted)
    if not ocr_confidence(extracted):
        print('Warning: low-confidence OCR result. Consider retaking the photo.')
    question_text = extracted.strip()

else:
    print('Invalid option.')

question_text


## 2. Data Normalisation & Quality Check (Section 4.2)

If you used image input, score OCR fidelity against a known reference string using 
`ocr_accuracy()` — this feeds the **Optical Character Recognition** row in Section 5.1.


In [ ]:
# Only meaningful for image input where you know the ground-truth text.
reference_text = ''  # e.g. '2x + 3 = 11'
if reference_text:
    stats = ocr_accuracy(reference_text, question_text)
    print(stats)


## 3. Spatial Mathematics Decoding + Semantic Inference (Section 4.3)

`run_pipeline()` tries the **local deterministic engine first** (arithmetic, linear/
quadratic equations, up-to-3×3 matrices — zero hallucination risk, works offline, 
natively speaks English/Hindi/Tamil). Only if that engine can't parse the question, or 
the target language isn't one it natively speaks, does the pipeline call 
`openai/gpt-oss-20b` on Groq (`temperature=0`, `<think>` tags stripped). The result is 
always tagged with which engine actually produced it.


In [ ]:
target_language = input('Translate the solution into which language? (English/Hindi/Tamil/...): ') or 'English'

result = run_pipeline(question_text, target_language=target_language, make_audio=False)
print('Engine used:', result.engine)
if result.notice:
    print('Notice:', result.notice)
print('\nSolution:\n', result.solution_text)

# The local engine now also covers abs(), log(), ln(), sin(), cos(), tan()
# (degrees) alongside arithmetic, linear/quadratic equations, and matrices --
# try: 'sin(30) + cos(60)', 'log(100)', 'abs(-7) + 2'


## 4. Matrix Output Compilation (Section 4.4)

Splits the solution into the Tactile Matrix Array (Braille preview, reusing the same 
lookup dict `engine.py`'s own local solver uses) and a slow, localized (`tld='co.in'`) 
`gTTS` audio track, played inline below.


In [ ]:
print('Braille preview:\n', result.braille_text)
print('\nBraille fidelity:', result.braille_fidelity)


In [ ]:
from IPython.display import Audio
audio_path = synthesize_audio(result.solution_text, target_language=target_language, out_path='solution.mp3')
Audio(audio_path, autoplay=False)


## 5. Experimental Results (Section 5.1) — real, offline-measured numbers

These numbers come from `benchmark.py`, run entirely offline against synthetic/
controlled data (no real classroom photos or live student trials were available in 
this environment — see the `methodology` field of each metric below for exactly how 
it was measured). Replace with real classroom/user-test data before citing these as 
human-subject results.


In [ ]:
!python benchmark.py


In [ ]:
import json
import pandas as pd

with open('benchmark_results.json') as f:
    bench = json.load(f)

results = pd.DataFrame([
    {'Metric': 'Optical Character Recognition',
     'Result': f"{bench['optical_character_recognition']['average_accuracy_pct']}% avg accuracy",
     'Samples': bench['optical_character_recognition']['sample_count']},
    {'Metric': 'Braille Translation Fidelity',
     'Result': f"{bench['braille_translation_fidelity']['aggregate_accuracy_pct']}% fidelity",
     'Samples': bench['braille_translation_fidelity']['sample_count']},
    {'Metric': 'Spatial Math Comprehension (proxy)',
     'Result': f"{bench['spatial_math_comprehension_proxy']['ambiguity_reduction_pct']} pt ambiguity reduction",
     'Samples': bench['spatial_math_comprehension_proxy']['sample_count']},
])
results


In [ ]:
import matplotlib.pyplot as plt

proxy = bench['spatial_math_comprehension_proxy']
fig, ax = plt.subplots()
labels = ['Naive Linear Reading', 'Local Spatial Decoder']
ambiguous_pct = [proxy['naive_ambiguous_pct'], proxy['structured_ambiguous_pct']]
ax.bar(labels, ambiguous_pct, color=['#999999', '#2b6cb0'])
ax.set_ylabel('Structurally Ambiguous Readings (%)')
ax.set_title('Structural Ambiguity: Linear Reader vs. Spatial Decoder (rule-based proxy)')
plt.show()


## 6. Worksheet (batch) mode

A real classroom lab processes a whole worksheet, not one question at a time. 
`run_worksheet()` solves a list of questions in one pass and reports what fraction 
were answered by the verified local engine vs. the LLM fallback — direct evidence for 
how much of a real assignment this framework can handle with zero LLM-math risk.


In [ ]:
from pipeline import run_worksheet

questions = [
    '2*x + 3 = 11',
    'sqrt(16) + 9',
    'x^2 - 5*x + 6 = 0',
    'sin(30) + cos(60)',
    '[[1,2],[3,4]]',
]
report = run_worksheet(questions, target_language=target_language, make_audio=False)
for i, out in enumerate(report['outputs'], 1):
    print(f"[{i}] ({out.engine}) {out.input_text} -> {out.solution_text.splitlines()[-1]}")
print('\nEngine breakdown:', report['engine_counts'])
print('Solved with zero LLM-math risk:', report['solved_without_llm_math_risk_pct'], '%')


## Notes / Limitations (Section 6)

- The on-screen Braille here is an uncontracted Grade-1 preview, not certified 
  Nemeth/UEB math code or Bharati Braille, and cannot raise physical tactile dots.
- The local deterministic engine covers arithmetic, linear/quadratic equations, and 
  up-to-3×3 matrices; anything beyond that scope routes to the cloud LLM and is 
  explicitly flagged `llm-fallback (unverified)` rather than presented as guaranteed-correct.
- `faster-whisper` downloads a small model file the first time it runs; after that it 
  works fully offline, unlike a cloud speech API.
- Section 5's numbers above are from a synthetic/rule-based benchmark, not a live 
  classroom trial — see each metric's `methodology` field in `benchmark_results.json`.
